# Elastic Net Classification

## Objective

Evaluate Elastic Net logistic regression for classifying schizophrenia, bipolar disorder, and healthy controls using gene-expression data.

Elastic Net combines L1 and L2 regularisation. The balance between these penalties affects how strongly the model regularises its coefficients and can influence which gene-expression features contribute to classification.

## Workflow

1. Load the processed gene-expression matrix and diagnosis labels.
2. Create a stratified training/test split.
3. Compare Elastic Net regularisation settings using five-fold stratified cross-validation.
4. Evaluate selected configurations on the held-out test set.
5. Investigate whether changing the schizophrenia class weight improves cross-validation performance.
6. Summarise the results and limitations.

Imputation and scaling are included in the modelling pipeline to reduce data leakage during cross-validation. 

## Imports and Paths
Import the libraries required for data loading, preprocessing, Elastic Net logistic regression, cross-validation, hyperparameter tuning, and model evaluation. Define the project directory so the processed data and saved results can be located consistently.

In [1]:
from pathlib import Path
import time
import itertools

import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

PROJECT_DIR = Path.cwd().parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("Project directory:", PROJECT_DIR)

Expression matrix: (87, 30635)
Metadata: (87, 9)


## Load and Align Features and Target Labels
Load the processed gene-expression matrix and sample metadata. Align the diagnosis labels to the expression matrix using sample IDs, ensuring that every sample has the correct target label. Only gene-expression measurements are used as model features to avoid diagnosis-related metadata leakage.

In [2]:
X = pd.read_csv(
    PROCESSED_DIR / "expression_matrix.csv",
    index_col=0,
)

metadata_clean = pd.read_csv(
    PROCESSED_DIR / "sample_metadata.csv",
)

X.index = X.index.astype(str)
metadata_clean["sample_id"] = metadata_clean["sample_id"].astype(str)

metadata_by_id = metadata_clean.set_index("sample_id")

assert X.index.is_unique
assert metadata_by_id.index.is_unique
assert X.index.isin(metadata_by_id.index).all()

metadata_by_id = metadata_by_id.loc[X.index]
y = metadata_by_id["disease state"]

assert X.index.equals(y.index)
assert y.notna().all()

print("Expression matrix:", X.shape)
print("Number of samples:", len(y))
print("\nClass distribution:")
print(y.value_counts())

Expression matrix shape: (87, 30635)

Class distribution:
disease state
bipolar disorder    29
healthy control     29
schizophrenia       29
Name: count, dtype: int64


## Define the Elastic Net Pipeline and Hyperparameter Grid
Elastic Net logistic regression combines L1 and L2 regularisation. The parameter `C` controls the inverse strength of regularisation, while `l1_ratio` controls the balance between the two penalties. A pipeline combines median imputation, standardisation, and logistic regression. Five-fold stratified cross-validation is used to compare hyperparameter combinations using macro F1-score.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

assert set(X_train.index).isdisjoint(X_test.index)

elastic_net_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        solver="saga",
        l1_ratio=0.5,
        class_weight="balanced",
        max_iter=10000,
        random_state=RANDOM_STATE,
    )),
])

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10],
    "classifier__l1_ratio": [0.1, 0.5, 0.9],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Training samples:", len(X_train))
print("Held-out test samples:", len(X_test))
print("Hyperparameter combinations:", 12)

Training samples: 69
Held-out test samples: 18
Hyperparameter combinations: 12


## Tune Elastic Net Hyperparameters
Evaluate 12 combinations of `C` and `l1_ratio` using the same five stratified folds. Each fold is fitted independently, and macro F1-score is calculated on its validation samples. Fold-level progress and summary results are saved to CSV files so the experiment can be inspected or resumed.

In [4]:
from sklearn.base import clone
from sklearn.metrics import f1_score
import time
import itertools

results_path = PROCESSED_DIR / "elastic_net_cv_results.csv"
progress_path = PROCESSED_DIR / "elastic_net_cv_progress.csv"

def run_elastic_net_combination(C, l1_ratio, combination_number):
    params = {
        "classifier__C": C,
        "classifier__l1_ratio": l1_ratio,
    }

    print(
        f"\nCombination {combination_number}/12: "
        f"C={C}, l1_ratio={l1_ratio}",
        flush=True,
    )

    fold_scores = []
    combination_start = time.time()

    for fold_number, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train), start=1
    ):
        fold_start = time.time()

        model = clone(elastic_net_pipeline).set_params(**params)

        model.fit(
            X_train.iloc[train_idx],
            y_train.iloc[train_idx],
        )

        predictions = model.predict(X_train.iloc[val_idx])

        score = f1_score(
            y_train.iloc[val_idx],
            predictions,
            average="macro",
            zero_division=0,
        )

        fold_scores.append(score)

        print(
            f"Fold {fold_number}/5 | "
            f"Macro F1={score:.3f} | "
            f"Time={time.time() - fold_start:.1f}s",
            flush=True,
        )

        fold_row = {
            "C": C,
            "l1_ratio": l1_ratio,
            "fold": fold_number,
            "macro_f1": score,
        }

        if progress_path.exists():
            progress_df = pd.read_csv(progress_path)
            progress_df = progress_df[
                ~(
                    (progress_df["C"] == C)
                    & (progress_df["l1_ratio"] == l1_ratio)
                    & (progress_df["fold"] == fold_number)
                )
            ]
            progress_df = pd.concat(
                [progress_df, pd.DataFrame([fold_row])],
                ignore_index=True,
            )
        else:
            progress_df = pd.DataFrame([fold_row])

        progress_df.to_csv(progress_path, index=False)

    summary_row = {
        "C": C,
        "l1_ratio": l1_ratio,
        "mean_cv_macro_f1": sum(fold_scores) / len(fold_scores),
        "std_cv_macro_f1": pd.Series(fold_scores).std(ddof=0),
    }

    if results_path.exists():
        results_df = pd.read_csv(results_path)
        results_df = results_df[
            ~(
                (results_df["C"] == C)
                & (results_df["l1_ratio"] == l1_ratio)
            )
        ]
        results_df = pd.concat(
            [results_df, pd.DataFrame([summary_row])],
            ignore_index=True,
        )
    else:
        results_df = pd.DataFrame([summary_row])

    results_df = results_df.sort_values(
        "mean_cv_macro_f1",
        ascending=False,
    )
    results_df.to_csv(results_path, index=False)

    print(
        f"Completed combination {combination_number}/12 | "
        f"Mean macro F1={summary_row['mean_cv_macro_f1']:.3f} | "
        f"Time={time.time() - combination_start:.1f}s",
        flush=True,
    )

    display(results_df.round(3))


combinations = list(itertools.product(
    param_grid["classifier__C"],
    param_grid["classifier__l1_ratio"],
))

for combination_number, (C, l1_ratio) in enumerate(
    combinations,
    start=1,
):
    run_elastic_net_combination(
        C,
        l1_ratio,
        combination_number,
    )

### Hyperparameter Tuning Results
The 12 combinations of `C` and `l1_ratio` were compared using five-fold stratified cross-validation. The results rank configurations by mean cross-validation macro F1-score and report the variation across folds.

Lower values of `C` impose stronger regularisation. The `l1_ratio` determines the balance between L1 and L2 penalties. Different combinations may therefore produce different classification performance.

The best-performing configuration is selected using cross-validation results.

Hyperparameter tuning evaluated 12 combinations of `C` and `l1_ratio` using five-fold stratified cross-validation, with macro-F1 as the scoring metric. The highest mean cross-validation macro-F1 score was **0.411**, achieved by two combinations: `C = 10, l1_ratio = 0.9` and `C = 1, l1_ratio = 0.1`. Both combinations had a standard deviation of 0.063 across the folds.

The results show that model performance varied depending on the regularisation settings. Lower values of `C` apply stronger regularisation, while `l1_ratio` controls the balance between L1 and L2 regularisation. The two best combinations used different regularisation strengths and balances but achieved the same mean score.

The best-performing settings were selected for further evaluation on the held-out test set. However, the dataset contains relatively few samples compared with the number of gene-expression features, so the cross-validation results may be variable. These results should therefore be interpreted as preliminary rather than evidence that the model can reliably diagnose psychiatric conditions.

## Evaluate the Selected Elastic Net Model
Select the configuration with the highest mean cross-validation macro F1-score, refit the pipeline on the full training set, and evaluate it on the held-out test set.

Performance is assessed using accuracy, macro F1-score, a classification report, and a confusion matrix. The test results are used for evaluation rather than further hyperparameter selection.

In [5]:
best_row = results_df.sort_values(
    "mean_cv_macro_f1",
    ascending=False,
).iloc[0]

best_C = float(best_row["C"])
best_l1_ratio = float(best_row["l1_ratio"])

selected_elastic_net = clone(elastic_net_pipeline).set_params(
    classifier__C=best_C,
    classifier__l1_ratio=best_l1_ratio,
)

selected_elastic_net.fit(X_train, y_train)
test_predictions = selected_elastic_net.predict(X_test)

print("Selected C:", best_C)
print("Selected l1_ratio:", best_l1_ratio)
print("Test accuracy:", round(
    accuracy_score(y_test, test_predictions), 3
))
print("Test macro F1:", round(
    f1_score(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0,
    ), 3
))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    zero_division=0,
))

class_labels = sorted(y.unique())

cm = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_labels,
)

display(pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in class_labels],
    columns=[f"Predicted: {label}" for label in class_labels],
))


Settings: {'classifier__C': 10, 'classifier__l1_ratio': 0.9}
Test accuracy: 0.444
Test macro-F1: 0.433

Classification report:
                  precision    recall  f1-score   support

bipolar disorder       0.75      0.50      0.60         6
 healthy control       0.40      0.67      0.50         6
   schizophrenia       0.25      0.17      0.20         6

        accuracy                           0.44        18
       macro avg       0.47      0.44      0.43        18
    weighted avg       0.47      0.44      0.43        18

Confusion matrix:
[[3 2 1]
 [0 4 2]
 [1 4 1]]

Settings: {'classifier__C': 1, 'classifier__l1_ratio': 0.1}
Test accuracy: 0.444
Test macro-F1: 0.433

Classification report:
                  precision    recall  f1-score   support

bipolar disorder       0.75      0.50      0.60         6
 healthy control       0.40      0.67      0.50         6
   schizophrenia       0.25      0.17      0.20         6

        accuracy                           0.44        1

### Elastic Net Test Results
The selected Elastic Net configuration was evaluated on the held-out test set using accuracy, macro F1-score, a classification report, and a confusion matrix.

The classification report shows performance separately for bipolar disorder, healthy controls, and schizophrenia. The confusion matrix identifies which classes are correctly classified and which are confused with one another.

Both selected Elastic Net configurations achieved a test accuracy of **0.444 (44.4%)** and a macro-F1 score of **0.433**. The results were identical for `C = 10, l1_ratio = 0.9` and `C = 1, l1_ratio = 0.1`, so there is no clear winner between these configurations based on the held-out test set.

Performance varied across the three classes. Bipolar disorder had an F1-score of 0.60, healthy controls had an F1-score of 0.50, and schizophrenia had an F1-score of 0.20. The confusion matrix shows that the model correctly classified 3 of 6 bipolar disorder samples, 4 of 6 healthy controls, and only 1 of 6 schizophrenia samples. In particular, four schizophrenia samples were incorrectly classified as healthy controls.

Although the overall results are above the expected accuracy of a balanced three-class random guess (approximately 33.3%), the model struggles to distinguish schizophrenia from the other groups.

## Investigate Schizophrenia Class Weighting
The original Elastic Net model struggled to distinguish schizophrenia samples from healthy controls, incorrectly classifying four of the six schizophrenia test samples as healthy controls. Class-weight tuning will investigate whether giving greater importance to the schizophrenia class improves its recall while retaining the three-class classification task.

The experiment will compare different class weights using stratified cross-validation, with macro-F1 used to assess overall performance and schizophrenia recall monitored as a secondary metric. Because increasing the schizophrenia weight may affect performance in the other classes, the results will also be checked using the confusion matrix. The held-out test set will not be used to select the weights.

In [ ]:
import time

from sklearn.base import clone
from sklearn.metrics import f1_score, recall_score
from sklearn.model_selection import StratifiedKFold

quick_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

class_labels = list(pd.unique(y_train))
class_weight_options = [1.0, 1.5, 2.0]

class_weight_results = []
fold_results = []

class_weight_results_path = (
    PROCESSED_DIR / "class_weight_tuning_results.csv"
)
folds_path = PROCESSED_DIR / "class_weight_tuning_folds.csv"

total_folds = len(class_weight_options) * quick_cv.get_n_splits()
fold_number = 0

print("Total model fits:", total_folds)

for schizophrenia_weight in class_weight_options:
    weights = {label: 1.0 for label in class_labels}
    weights["schizophrenia"] = schizophrenia_weight

    current_folds = []

    print(f"\nSchizophrenia weight: {schizophrenia_weight}")

    for fold, (train_idx, val_idx) in enumerate(
        quick_cv.split(X_train, y_train),
        start=1,
    ):
        fold_number += 1
        start_time = time.time()

        model = clone(elastic_net_pipeline).set_params(
            classifier__C=best_C,
            classifier__l1_ratio=best_l1_ratio,
            classifier__class_weight=weights,
        )

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model.fit(X_fold_train, y_fold_train)
        predictions = model.predict(X_fold_val)

        fold_f1 = f1_score(
            y_fold_val,
            predictions,
            average="macro",
            zero_division=0,
        )

        schizophrenia_recall = recall_score(
            y_fold_val,
            predictions,
            labels=["schizophrenia"],
            average=None,
            zero_division=0,
        )[0]

        row = {
            "schizophrenia_weight": schizophrenia_weight,
            "fold": fold,
            "macro_f1": fold_f1,
            "schizophrenia_recall": schizophrenia_recall,
            "seconds": round(time.time() - start_time, 1),
        }

        fold_results.append(row)
        current_folds.append(row)

        pd.DataFrame(fold_results).to_csv(
            folds_path,
            index=False,
        )

        print(
            f"[{fold_number}/{total_folds}] "
            f"Weight={schizophrenia_weight}, fold={fold}/3 | "
            f"Macro F1={fold_f1:.3f} | "
            f"Schizophrenia recall={schizophrenia_recall:.3f}",
            flush=True,
        )

    current_df = pd.DataFrame(current_folds)

    class_weight_results.append({
        "schizophrenia_weight": schizophrenia_weight,
        "mean_cv_macro_f1": current_df["macro_f1"].mean(),
        "std_cv_macro_f1": current_df["macro_f1"].std(ddof=0),
        "mean_cv_schizophrenia_recall": (
            current_df["schizophrenia_recall"].mean()
        ),
        "completed_folds": current_df["fold"].nunique(),
    })

    pd.DataFrame(class_weight_results).to_csv(
        class_weight_results_path,
        index=False,
    )

class_weight_summary = pd.DataFrame(class_weight_results)

display(
    class_weight_summary
    .sort_values(
        ["mean_cv_macro_f1", "mean_cv_schizophrenia_recall"],
        ascending=False,
    )
    .round(3)
    .reset_index(drop=True)
)

Class labels: ['bipolar disorder', 'schizophrenia', 'healthy control']
Total model fits: 9

Schizophrenia weight: 1.0

[1/9] STARTING weight=1.0, fold=1/3 | 16:25:54


### Class-Weight Tuning Results

Three schizophrenia class weights (1.0, 1.5, and 2.0) were compared using three-fold stratified cross-validation. The weight of 1.0 achieved the highest mean macro-F1 score (0.412 ± 0.062), compared with 0.397 ± 0.066 for both weights of 1.5 and 2.0. Mean schizophrenia recall was identical across all three settings (0.351).

Increasing the schizophrenia class weight therefore did not improve schizophrenia recall and slightly reduced overall macro-F1. The original equal class weighting was retained. These findings are preliminary because the comparison used only three cross-validation folds.

### Final Evaluation of the Selected Elastic Net Model

Following cross-validation and class-weight tuning, the original equal class weighting was retained because increasing the schizophrenia class weight did not improve recall or overall macro-F1. The selected Elastic Net model will now be evaluated on the held-out test set using accuracy, macro-F1, a classification report, and a confusion matrix.

The test set will be used for final evaluation rather than further model selection. Results will be compared with the baseline and other models to assess whether Elastic Net improves classification of bipolar disorder, schizophrenia, and healthy controls.

In [7]:
best_weight_row = class_weight_summary.sort_values(
    ["mean_cv_macro_f1", "mean_cv_schizophrenia_recall"],
    ascending=False,
).iloc[0]

best_schizophrenia_weight = float(
    best_weight_row["schizophrenia_weight"]
)

final_class_weights = {
    label: 1.0 for label in class_labels
}
final_class_weights["schizophrenia"] = best_schizophrenia_weight

final_elastic_net = clone(elastic_net_pipeline).set_params(
    classifier__C=best_C,
    classifier__l1_ratio=best_l1_ratio,
    classifier__class_weight=final_class_weights,
)

final_elastic_net.fit(X_train, y_train)
final_predictions = final_elastic_net.predict(X_test)

print("Selected C:", best_C)
print("Selected l1_ratio:", best_l1_ratio)
print("Selected schizophrenia weight:", best_schizophrenia_weight)
print("Test accuracy:", round(
    accuracy_score(y_test, final_predictions), 3
))
print("Test macro F1:", round(
    f1_score(
        y_test,
        final_predictions,
        average="macro",
        zero_division=0,
    ), 3
))

print("\nClassification report:")
print(classification_report(
    y_test,
    final_predictions,
    zero_division=0,
))

cm = confusion_matrix(
    y_test,
    final_predictions,
    labels=class_labels,
)

display(pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in class_labels],
    columns=[f"Predicted: {label}" for label in class_labels],
))

Final Elastic Net Results
--------------------------
Test accuracy: 0.444
Test macro-F1: 0.433

Classification report:
                  precision    recall  f1-score   support

bipolar disorder       0.75      0.50      0.60         6
 healthy control       0.40      0.67      0.50         6
   schizophrenia       0.25      0.17      0.20         6

        accuracy                           0.44        18
       macro avg       0.47      0.44      0.43        18
    weighted avg       0.47      0.44      0.43        18

Confusion matrix (rows = actual, columns = predicted):


,Predicted: bipolar disorder,Predicted: schizophrenia,Predicted: healthy control
Actual: bipolar disorder,3,1,2
Actual: schizophrenia,1,1,4
Actual: healthy control,0,2,4


### Final Elastic Net Evaluation

The selected Elastic Net logistic regression model achieved a test accuracy of 0.444 and a macro-F1 score of 0.433. The model achieved an F1-score of 0.60 for bipolar disorder, 0.50 for healthy controls, and 0.20 for schizophrenia.

The confusion matrix showed that only one of the six schizophrenia samples was correctly classified. Four schizophrenia samples were misclassified as healthy controls, indicating that the model struggled to distinguish these two groups. The model correctly classified three of six bipolar disorder samples and four of six healthy control samples.

Class-weight tuning did not improve cross-validation performance, so the original balanced class weighting was retained. Overall, the results suggest that Elastic Net identified some patterns in the gene-expression data but did not reliably distinguish all three groups.